# 01 — Gerando nossa base de dados "relacional"

Todo projeto de carga de dados começa com um banco relacional (ou um conjunto de
planilhas/CSVs vindos de um sistema legado). Neste notebook vamos **simular** esse
cenário: um banco fictício brasileiro que registra clientes, seus dados de contato
e as transações que fazem (Pix, boleto, compra no cartão, saque, depósito).

Vamos gerar as tabelas com a biblioteca [Faker](https://faker.readthedocs.io/), mas
com uma pegadinha: vamos **injetar de propósito** alguns padrões de fraude
(identidades reutilizadas, Pix repetidos para contas-laranja). Não é assim que
fraude aparece nos dados no mundo real — lá você não sabe onde procurar — mas é
assim que fraude aparece quando o objetivo é *aprender a procurar*: precisamos
garantir que os padrões existam para poder encontrá-los no notebook 04.

**O que vamos gerar:**

| Tabela | Colunas | O que representa |
|---|---|---|
| `clientes` | `cpf`, `nome`, `data_cadastro`, `rg`, `email`, `telefone` (+ 2 colunas de gabarito) | cadastro de pessoas |
| `bancos` | `codigo`, `nome` | bancos de destino (pagamento de boleto) |
| `empresas` | `cnpj`, `nome` | lojas, correspondentes e agentes |
| `transacoes` | `transacao_id`, `tipo`, `valor`, `step`, `ts`, `origem_cpf`, `destino_id` | cada Pix, boleto, compra, saque ou depósito |

Quatro tabelas, do jeito que um sistema bancário real guardaria — e cada uma
identificada pelo documento que a entidade já tem no mundo real: pessoa por
**CPF**, empresa por **CNPJ**, banco por **código de compensação**.

Repare que o cadastro do cliente é uma linha só, com RG, e-mail e telefone como
**colunas**. Guarde esse detalhe: é justamente ele que vai mudar de forma no
notebook 02 — e que vai tornar a fraude visível.

In [ ]:
!pip install -q faker pandas

## Onde ficam os CSVs

Este notebook detecta sozinho onde está rodando:

- **No Google Colab** — monta o seu Google Drive e usa a pasta
  `workshop-neo4j-csv` dentro dele. Assim os arquivos **sobrevivem** ao fim da
  sessão: o Colab apaga o disco local quando o runtime é reciclado, mas o que está
  no Drive fica. É o que permite gerar os dados hoje e recarregar amanhã.
- **Localmente** — usa a pasta `data/` ao lado do notebook.

No Colab, a célula abaixo vai abrir um pedido de autorização do Google. É o próprio
Colab pedindo acesso ao seu Drive; sem isso ele não consegue gravar lá.

In [ ]:
import os

try:
    from google.colab import drive
    EM_COLAB = True
except ImportError:
    EM_COLAB = False

if EM_COLAB:
    drive.mount("/content/drive")
    # o Drive do usuário fica em MyDrive — escrever na raiz da montagem
    # (/content/drive) não sincroniza com o Drive de verdade
    raiz_drive = "/content/drive/MyDrive"
    if not os.path.isdir(raiz_drive):
        raiz_drive = "/content/drive/My Drive"   # nome antigo, com espaço
    PASTA_DADOS = os.path.join(raiz_drive, "workshop-neo4j-csv")
else:
    PASTA_DADOS = "data"

os.makedirs(PASTA_DADOS, exist_ok=True)

print(f"Ambiente: {'Google Colab (dados no seu Drive)' if EM_COLAB else 'local'}")
print(f"Pasta dos CSVs: {PASTA_DADOS}")

## 0. Parâmetros — edite aqui para controlar o tamanho da base

Esses são os únicos números que você precisa tocar neste notebook.

O **AuraDB Free** aceita até **200.000 nós e 400.000 relacionamentos** — e esse
limite é imposto pelo servidor: ao passar dele, a carga falha com
`You have exceeded the logical size limit`. Os valores padrão abaixo usam cerca de
**90% da cota de nós**, deixando margem para os relacionamentos extras que o
notebook 04 cria durante a investigação.

Se quiser reduzir (para um teste rápido) ou aumentar (numa instância paga ou local),
mude `N_CLIENTES` e/ou `N_TRANSACOES` e rode a célula de estimativa logo abaixo —
ela avisa se a configuração passa do limite **antes** de você gastar tempo gerando
e carregando os dados.

> ⏱️ Com os valores padrão, a carga do notebook 02 leva alguns minutos. Se você só
> quer percorrer o material rápido, use `N_CLIENTES = 1_000` e
> `N_TRANSACOES = 15_000`: tudo funciona igual, a fraude continua detectável e a
> carga cai para menos de um minuto.

In [ ]:
import random
import os
import json
import pandas as pd
from faker import Faker

# ------------------------------------------------------------------
# PARÂMETROS — edite livremente. Os valores padrão usam ~90% da cota de nós do
# AuraDB Free (200.000 nós / 400.000 relacionamentos).
# ------------------------------------------------------------------
SEED = 42                    # fixo = a base gerada é sempre a mesma
N_CLIENTES = 10_000
N_BANCOS = 8
N_EMPRESAS = 200
N_TRANSACOES = 140_000
N_ANEIS_FRAUDE = 25          # grupos de clientes que compartilham identificadores
TAMANHO_ANEL = (3, 6)        # min/max de clientes por anel
N_CONTAS_LARANJA = 15        # clientes que vão receber Pix suspeitos
# ------------------------------------------------------------------

random.seed(SEED)
fake = Faker("pt_BR")
Faker.seed(SEED)

### Conferindo a volumetria antes de gerar

Cada cliente vai gerar, no máximo, 4 nós (`Cliente` + `RG` + `Email` +
`Telefone`) e 3 relacionamentos. Cada transação gera 1 nó (`Transacao`) e 2
relacionamentos (`REALIZOU`, `PARA`). A conta abaixo usa o pior caso (nenhum
identificador compartilhado) — na prática o número real de nós fica um pouco
*menor*, porque os anéis de fraude fazem identificadores serem reaproveitados em
vez de criar nós novos.

In [ ]:
LIMITE_NOS_AURA_FREE = 200_000
LIMITE_RELS_AURA_FREE = 400_000

def estimar_volumetria(n_clientes, n_bancos, n_empresas, n_transacoes_base, n_aneis_fraude):
    # pior caso de Pix extras injetados (padrão "conta-laranja"):
    # até 4 contas-laranja por fraudador, até 10 Pix por conta-laranja
    max_pix_extra = n_aneis_fraude * 4 * 10
    transacoes_totais = n_transacoes_base + max_pix_extra

    nos = (
        n_clientes          # :Cliente
        + n_clientes * 3    # :RG + :Email + :Telefone (limite superior, sem nenhum compartilhamento)
        + n_bancos          # :Banco
        + n_empresas        # :Empresa
        + transacoes_totais # :Transacao
    )
    rels = (
        n_clientes * 3          # TEM_RG + TEM_EMAIL + TEM_TELEFONE
        + transacoes_totais * 2 # REALIZOU + PARA
    )
    return nos, rels

nos_estimados, rels_estimados = estimar_volumetria(
    N_CLIENTES, N_BANCOS, N_EMPRESAS, N_TRANSACOES, N_ANEIS_FRAUDE
)

print(f"Estimativa (pior caso): ~{nos_estimados:,} nós e ~{rels_estimados:,} relacionamentos")
print(f"Limite do AuraDB Free:  {LIMITE_NOS_AURA_FREE:,} nós e {LIMITE_RELS_AURA_FREE:,} relacionamentos")

if nos_estimados > LIMITE_NOS_AURA_FREE or rels_estimados > LIMITE_RELS_AURA_FREE:
    print("\n❌ Essa configuração ESTOURA o limite do AuraDB Free — a carga vai falhar.")
    print("   Reduza N_CLIENTES e/ou N_TRANSACOES antes de continuar.")
else:
    uso_nos = 100 * nos_estimados / LIMITE_NOS_AURA_FREE
    uso_rels = 100 * rels_estimados / LIMITE_RELS_AURA_FREE
    print(f"\n✅ Cabe no AuraDB Free — usando {uso_nos:.0f}% da cota de nós "
          f"e {uso_rels:.0f}% da de relacionamentos.")

## 1. Clientes — e os anéis de fraude

Primeiro criamos cada cliente com todos os campos **únicos**. Depois escolhemos
alguns grupos ("anéis") e forçamos todo mundo do grupo a compartilhar o **mesmo**
valor em um ou dois campos.

É o padrão que o [guia oficial de detecção de fraude do
Neo4j](https://github.com/neo4j-graph-examples/fraud-detection) usa para achar
fraude de primeira parte (*first-party fraud*): a mesma pessoa (ou o mesmo grupo)
abrindo várias contas com identidades ligeiramente diferentes, mas reaproveitando
dados de contato.

Repare que, na tabela, essa fraude fica **invisível**: são 10.000 linhas de
aparência normal, e pouco mais de 1% delas pertence a um anel. Nada na estrutura de
`clientes` sugere que algumas dessas linhas são a mesma pessoa.

In [ ]:
clientes = []
for _ in range(N_CLIENTES):
    clientes.append({
        "cpf": fake.unique.cpf(),          # chave primária: o CPF é validado e único
        "nome": fake.name(),
        "data_cadastro": fake.date_between(start_date="-3y", end_date="today").isoformat(),
        "rg": fake.unique.rg(),            # campo aberto: sem unicidade nacional
        "email": fake.unique.email(),
        "telefone": fake.unique.phone_number(),
        # colunas de gabarito — preenchidas nas células de injeção mais abaixo
        "gabarito_anel": "",
        "gabarito_laranja": False,
    })

print(f"{len(clientes)} clientes gerados")
pd.DataFrame(clientes).head()

Agora a injeção. Note **quais** campos os anéis compartilham: RG, e-mail e
telefone — nunca o CPF.

Isso não é detalhe de implementação, é o que torna o cenário plausível. O CPF é
validado e único: um sistema bancário real recusaria um segundo cadastro com o
mesmo CPF, e é por isso que ele serve como chave primária. Já o RG **não tem
unicidade nacional** (é emitido por estado, com formatos diferentes) e quase nunca
é validado no cadastro — é o campo "aberto" clássico. E-mail e telefone, idem.

É justamente nesses campos frouxos que a fraude de identidade se instala.

In [ ]:
# índice auxiliar para achar um cliente pelo CPF
por_cpf = {c["cpf"]: c for c in clientes}

cpfs_embaralhados = [c["cpf"] for c in clientes]
random.shuffle(cpfs_embaralhados)

aneis_de_fraude = []
campos_por_anel = []
cursor = 0
for _ in range(N_ANEIS_FRAUDE):
    tamanho = random.randint(*TAMANHO_ANEL)
    grupo = cpfs_embaralhados[cursor: cursor + tamanho]
    cursor += tamanho
    if len(grupo) < 2:
        continue

    # que campo(s) frouxo(s) esse anel vai compartilhar — o CPF nunca entra aqui
    campos = random.choice([["rg"], ["email"], ["telefone"], ["rg", "email"]])
    valores_compartilhados = {
        "rg": fake.rg(),
        "email": fake.email(),
        "telefone": fake.phone_number(),
    }

    for cpf in grupo:
        for campo in campos:
            por_cpf[cpf][campo] = valores_compartilhados[campo]

    aneis_de_fraude.append(grupo)
    campos_por_anel.append(campos)

    # gabarito: marca no próprio cadastro quem pertence a este anel.
    # Isso NÃO existiria num banco real — é um campo de conferência, que só
    # existe porque estes dados são sintéticos. Vai virar propriedade no grafo
    # (notebook 02) para o notebook 04 poder medir o acerto dos algoritmos.
    for cpf in grupo:
        por_cpf[cpf]["gabarito_anel"] = len(aneis_de_fraude)

clientes_em_aneis = sum(len(a) for a in aneis_de_fraude)
print(f"{len(aneis_de_fraude)} anéis injetados, somando {clientes_em_aneis} clientes "
      f"({100 * clientes_em_aneis / N_CLIENTES:.1f}% da base).\n")

print("Detalhe dos 3 primeiros (CPFs distintos, campo frouxo repetido):\n")
for i, (anel, campos) in enumerate(zip(aneis_de_fraude[:3], campos_por_anel[:3]), 1):
    print(f"  Anel {i} ({len(anel)} clientes) compartilha: {', '.join(campos)}")
    for cpf in anel:
        c = por_cpf[cpf]
        valores = "  ".join(f"{campo}={c[campo]}" for campo in campos)
        print(f"      CPF {cpf}   {valores}")
    print()

resumo = {}
for campos in campos_por_anel:
    chave = "+".join(campos)
    resumo[chave] = resumo.get(chave, 0) + 1
print("Campos compartilhados, por anel:", resumo)

## 2. Bancos e empresas

Esses são os "destinos" possíveis de uma transação. `empresas` representa tanto
lojas quanto agentes e correspondentes bancários (lotéricas, por exemplo), que no
Brasil processam saque, depósito e pagamento no mesmo balcão.

Repare nas chaves primárias: usamos os identificadores que essas entidades já têm
no mundo real, em vez de inventar um id sequencial. Empresa se identifica por
**CNPJ**; banco, pelo **código de compensação** de 3 dígitos (o mesmo que você
digita numa transferência: 001, 341, 237…).

Isso vai importar no notebook 02: o CNPJ é um identificador nacional, exatamente
como o CPF do cliente. E, ainda assim, os dois vão ser modelados de formas
opostas no grafo — por um motivo que vale a discussão.

In [ ]:
# Códigos de compensação plausíveis (3 dígitos), como os bancos brasileiros usam
codigos_banco = random.sample([f"{n:03d}" for n in range(1, 400)], N_BANCOS)
bancos = [
    {"codigo": codigo, "nome": fake.company() + " Banco"}
    for codigo in codigos_banco
]

empresas = [
    {"cnpj": fake.unique.cnpj(), "nome": fake.company()}
    for _ in range(N_EMPRESAS)
]

print(f"{len(bancos)} bancos, {len(empresas)} empresas")
pd.DataFrame(empresas).head()

## 3. Transações

No cenário brasileiro, cada transação tem um `tipo` — `Pix`, `Boleto`, `Compra`,
`Saque` ou `Deposito` — e o tipo determina **quem pode ser o destino**: uma chave
estrangeira polimórfica clássica (o destino pode estar em três tabelas diferentes,
dependendo do tipo).

Guarde essa ideia também: no notebook 02 ela vai ser o segundo lugar onde o
modelo relacional trava e o grafo passa reto.

In [ ]:
TIPOS_TRANSACAO = {
    # tipo: (tabela de destino, peso na distribuição)
    "Pix": ("clientes", 0.35),
    "Compra": ("empresas", 0.30),
    "Deposito": ("empresas", 0.15),
    "Boleto": ("bancos", 0.10),
    "Saque": ("empresas", 0.10),
}

tipos = list(TIPOS_TRANSACAO)
pesos = [TIPOS_TRANSACAO[t][1] for t in tipos]

todos_ids_clientes = [c["cpf"] for c in clientes]

transacoes = []
tx_id = 1
for _ in range(N_TRANSACOES):
    tipo = random.choices(tipos, weights=pesos)[0]
    tabela_destino, _ = TIPOS_TRANSACAO[tipo]
    origem = random.choice(todos_ids_clientes)

    if tabela_destino == "empresas":
        destino_id = random.choice(empresas)["cnpj"]
    elif tabela_destino == "bancos":
        destino_id = random.choice(bancos)["codigo"]
    else:  # clientes (Pix)
        destino_id = random.choice([cid for cid in todos_ids_clientes if cid != origem])

    step = random.randint(1, 30)
    transacoes.append({
        "transacao_id": f"T{tx_id:06d}",
        "tipo": tipo,
        "valor": round(random.uniform(10, 5000), 2),
        "step": step,
        "ts": step * 3600,
        "origem_cpf": origem,
        "destino_id": destino_id,
        "fraude_real": False,
    })
    tx_id += 1

print(f"{len(transacoes)} transações \"normais\" geradas")

## 4. Injetando o padrão de "conta-laranja"

Agora a segunda pegadinha, adaptada ao golpe mais comum do Pix no Brasil: pegamos
um representante de cada anel de fraude (nosso "fraudador") e fazemos ele enviar
Pix, repetidas vezes e em valores altos, para um grupo separado de clientes — as
**contas-laranja**.

Essas contas não têm nenhuma identidade compartilhada — o truque do notebook 04
não vai funcionar nelas. A única pista que sobra é o *padrão de transação*: muitos
Pix, de valor alto, sempre vindos das mesmas poucas origens.

In [ ]:
fraudadores = [anel[0] for anel in aneis_de_fraude]
candidatos_conta_laranja = [cid for cid in todos_ids_clientes if cid not in fraudadores]
contas_laranja = random.sample(candidatos_conta_laranja, N_CONTAS_LARANJA)

for conta in contas_laranja:
    por_cpf[conta]["gabarito_laranja"] = True   # idem: campo de conferência

for fraudador in fraudadores:
    contas_do_fraudador = random.sample(contas_laranja, k=random.randint(2, 4))
    for conta in contas_do_fraudador:
        n_pix = random.randint(4, 10)
        for _ in range(n_pix):
            step = random.randint(1, 30)
            transacoes.append({
                "transacao_id": f"T{tx_id:06d}",
                "tipo": "Pix",
                "valor": round(random.uniform(2000, 15000), 2),
                "step": step,
                "ts": step * 3600,
                "origem_cpf": fraudador,
                "destino_id": conta,
                "fraude_real": True,
            })
            tx_id += 1

random.shuffle(transacoes)
n_fraudes = sum(t["fraude_real"] for t in transacoes)
print(f"Total de transações: {len(transacoes)} ({n_fraudes} marcadas como fraude real no gabarito)")

## 5. Salvando os CSVs

Isso é o que, na vida real, seria o resultado de um `SELECT * FROM tabela` (ou de
um export do seu DBA). São esses quatro arquivos que vamos carregar no Neo4j no
próximo notebook — gravados na pasta que a célula de ambiente definiu (seu Drive,
se você está no Colab).

In [ ]:
def salvar(nome, registros):
    df = pd.DataFrame(registros)
    caminho = os.path.join(PASTA_DADOS, f"{nome}.csv")
    df.to_csv(caminho, index=False)
    print(f"{caminho}: {len(df)} linhas, {len(df.columns)} colunas")
    return df

df_clientes = salvar("clientes", clientes)
df_bancos = salvar("bancos", bancos)
df_empresas = salvar("empresas", empresas)
df_transacoes = salvar("transacoes", transacoes)

### O gabarito viaja dentro do CSV

Repare nas duas últimas colunas de `clientes.csv`: `gabarito_anel` e
`gabarito_laranja`. Elas **não existiriam num banco de verdade** — nenhum cadastro
tem um campo "esta pessoa é fraudadora". Estão aqui porque os dados são sintéticos e
precisamos de um jeito de **medir** se os algoritmos do notebook 04 acertaram.

Elas vão virar propriedades no grafo (notebook 02) e são usadas só na última seção
do notebook 04, para comparar o que foi detectado com o que foi plantado. Nenhuma
consulta de detecção olha para elas.

In [ ]:
conferencia = df_clientes[
    (df_clientes["gabarito_anel"] != "") | (df_clientes["gabarito_laranja"])
]
print(f"{len(conferencia)} clientes marcados no gabarito:")
print(f"  em anéis de fraude: {(df_clientes['gabarito_anel'] != '').sum()}")
print(f"  contas-laranja:     {df_clientes['gabarito_laranja'].sum()}")
print(f"  ambos:              {((df_clientes['gabarito_anel'] != '') & (df_clientes['gabarito_laranja'])).sum()}")
conferencia.head()

### A fraude está lá — e você não a vê

Antes de sair deste notebook, olhe a tabela de clientes de novo. Ela tem 1.000
linhas de aparência absolutamente normal:

In [ ]:
df_clientes.head(10)

Nenhuma coluna diz "suspeito". Nenhuma linha se destaca — cada uma tem seu CPF
próprio, válido e distinto. E, ainda assim, plantamos 25 anéis de identidades
reaproveitadas e 15 contas-laranja ali dentro, diluídos em 10.000 cadastros.

Para achar os anéis em SQL você precisaria desconfiar **de antemão** e escrever um
`GROUP BY rg HAVING COUNT(*) > 1`, depois outro para `email`, depois outro para
`telefone` — uma consulta por coluna, e só encontrando o que você já suspeitava
procurar.

**Próximo passo:** o notebook `02_modelagem_e_carga.ipynb`, onde essas colunas vão
virar nós — e o reaproveitamento vai deixar de ser algo que você precisa procurar
para se tornar algo que salta aos olhos.